# CCA-F Capstone: the Nashville Music-Fest Ops Agent

> **CCA-F Capstone - all four code domains in one session.** This closes the `agents_api/` set. One themed **Managed Agent** session touches **Domain 1 - Agentic (27%)** (the create -> send -> stream -> idle -> teardown loop), **Domain 2 - Tools (18%)** (one custom tool answered with a structured result), **Domain 4 - Structured output (20%)** (JSON out of a turn, null-if-not-stated), and **Domain 5 - Context and escalation (15%)**. The one trap this capstone drills: read **`session.status_idle`** and **`stop_reason`** to know a turn is done, then answer the exact `tool_use` the model requested by its **`event_ids`**.

Theme: an **ops agent for a Nashville music-festival weekend** - stage schedule, a venue-capacity lookup tool, structured incident triage, and an escalation rule. We stand the agent up, run one turn that exercises all four domains, then always tear down.

**Voice and scope note.** These cells make **live, billable, beta-gated** calls (header `managed-agents-2026-04-01`). This notebook is authored to be read and AST-checked, not executed here. Run it only on a key with Managed Agents beta access, and let the teardown cell run.

### 1. Install dependencies

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

### 2. Load environment variables from .env file

The client reads `ANTHROPIC_API_KEY` from `examples/.env`. Copy `.env.example` to `.env` and fill it in. The key must carry **Managed Agents beta** access, or the create call returns a 403.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads examples/.env into os.environ

### 3. Create an API client, the model, and the beta header

**`MODEL`** stays on **Haiku 4.5** (the repo default). **`BETA`** is the Managed Agents beta header, passed on every `client.beta.*` call. We also sketch the one **custom tool** the agent may call - a venue-capacity lookup - as a JSON schema. **Custom tools** are the ones you define and answer yourself, distinct from built-in or MCP tools.

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-4-5"          # repo default; Sonnet only where reasoning depth earns it
BETA = "managed-agents-2026-04-01"  # Managed Agents beta header, passed on every beta call

# Teardown guard. Default OFF so "Run All" in class leaves the agent and session
# live for Console inspection. Headless smoke tests set ARCHIVE_ON_RUN=1 to force
# cleanup so no billable resource dangles in CI. See the final cell.
RUN_TEARDOWN = os.environ.get("ARCHIVE_ON_RUN", "0") == "1"

# Domain 2: one custom tool the agent may call. We answer it ourselves later.
VENUE_CAPACITY_TOOL = {
    "type": "custom",
    "name": "venue_capacity",
    "description": (
        "Look up the seated capacity for a Nashville music-festival venue. "
        "Returns a hard number; the agent must not guess capacity."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "venue": {
                "type": "string",
                "description": "Venue name, e.g. 'The Ryman' or 'Third Man Blue Room'.",
            }
        },
        "required": ["venue"],
    },
}

print(f"Client ready. Model={MODEL}. Beta={BETA}.")

### 4. Stand up the agent, environment, and session

Three creates, in order. The **agent** carries the versioned **`system`** prompt and its **`tools`**. The **environment** is the scratch runtime. The **session** binds an agent to an environment and holds turn memory **server-side**, so you never resend prior turns.

The `system` prompt folds two exam behaviors into the agent's recipe: **Domain 4** (return strict JSON, `null` when a field is not stated) and **Domain 5** (escalate on incident **shape**, not on how loud the report sounds).

In [ ]:
SYSTEM = (
    "You are the ops agent for a Nashville music-festival weekend. You run the stage "
    "schedule and triage venue incidents.\n"
    "- For any capacity question, call the venue_capacity tool. Never guess a number.\n"
    "- When asked for structured output, reply with STRICT JSON only, no prose. Use null "
    "for any field the report does not state; do not invent values.\n"
    "- Triage on incident SHAPE, not volume. A safety or crowd-flow blocker escalates to a "
    "human even when reported calmly; a routine seating question stays self-serve even when "
    "reported in a panic. Emit an escalate boolean in the triage JSON."
)

# 1. Agent: model + name required; system + tools optional.
agent = client.beta.agents.create(
    model={"id": MODEL},
    name="music-fest-ops",
    system=SYSTEM,
    tools=[VENUE_CAPACITY_TOOL],
    betas=[BETA],
)

# 2. Environment: the scratch runtime.
env = client.beta.environments.create(name="music-fest-scratch", betas=[BETA])

# 3. Session: binds agent to environment; holds turn memory server-side.
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    betas=[BETA],
)

print(f"agent={agent.id}\nenv={env.id}\nsession={session.id}")

### 5. Run one turn that exercises all four domains

Send a **user turn** that forces every behavior in one request: a capacity question (Domain 2 tool), a calmly worded crowd-flow blocker, and a demand for triage JSON (Domains 4 and 5).

Then **stream events** until **`session.status_idle`**. The **loop** is Domain 1: you stop on the idle event, not on the first `agent.message`. When the agent requests the `venue_capacity` tool, the idle event's **`stop_reason.event_ids`** point at the exact `agent.custom_tool_use` to answer. We answer with a **`user.custom_tool_result`** event keyed by **`custom_tool_use_id`** (built-in and MCP tools would use `user.tool_result` with `tool_use_id` instead), then stream again to the final idle.

> **Gotcha.** Answer the tool_use the model actually requested, found on the idle event's `stop_reason.event_ids`. Do not fabricate an id, and do not stop the loop on the first message you see.

In [ ]:
import json

# A local capacity book so we can answer the custom tool ourselves (Domain 2).
CAPACITY_BOOK = {"the ryman": 2362, "third man blue room": 250}


def run_until_idle(session_id):
    """Stream events until the session goes idle. Return the idle event.

    Domain 1: the loop stops on `session.status_idle`, never on the first
    `agent.message`. The returned event carries `stop_reason`, whose
    `event_ids` name the tool_use blocks awaiting an answer.
    """
    idle = None
    for event in client.beta.sessions.events.stream(session_id, betas=[BETA]):
        if event.type == "agent.message":
            print("agent:", event)
        elif event.type == "session.status_idle":
            idle = event
            break
        elif event.type == "session.status_terminated":
            raise RuntimeError("session terminated before idle")
    return idle


# The user turn: capacity lookup + a calmly worded crowd-flow blocker + a JSON demand.
client.beta.sessions.events.send(
    session.id,
    events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "Two things. First, what is the seated capacity of The Ryman for "
                "tonight's 9pm headliner set? Second, a volunteer just mentioned - very "
                "calmly - that the east stairwell exit is chained shut during the sold-out "
                "block. Return ONLY triage JSON with keys: venue (string), capacity "
                "(integer or null), incident (string), shape (one of 'safety', "
                "'crowd_flow', 'seating'), escalate (boolean), refund_owed (integer or "
                "null)."
            ),
        }],
    }],
    betas=[BETA],
)

idle = run_until_idle(session.id)

# Domain 2: if the model asked for the tool, answer the exact event_ids on stop_reason.
tool_event_ids = getattr(idle.stop_reason, "event_ids", None) or []
if tool_event_ids:
    results = []
    for use_id in tool_event_ids:
        # In a real app, read the requested venue from the tool_use block; here the
        # book has one seated hall, so we answer with its number.
        capacity = CAPACITY_BOOK["the ryman"]
        results.append({
            "type": "user.custom_tool_result",     # custom tools use this event type
            "custom_tool_use_id": use_id,          # ...keyed by custom_tool_use_id
            "content": [{"type": "text", "text": json.dumps({"capacity": capacity})}],
            "is_error": False,
        })
    client.beta.sessions.events.send(session.id, events=results, betas=[BETA])
    idle = run_until_idle(session.id)  # stream again to the final idle

print("final stop_reason:", idle.stop_reason)

### 6. Always tear down (guarded)

**Archive the session, then the agent.** A live session is a billable runtime; leaving it open leaks cost. There is **no `agents.delete`** on this surface - **`archive`** is the teardown verb. In production, wrap the run in `try/finally` so this path always executes (see Q5 below).

**This cell is guarded by `RUN_TEARDOWN`** (set in the client cell above). It defaults to **OFF** so a "Run All" during class leaves the agent and session **live** for Console inspection. To archive, flip the switch and re-run this one cell, or set `ARCHIVE_ON_RUN=1` before launching (how the smoke tests force cleanup). The guard decides *whether* to archive; a `try/finally` still guarantees this cell runs.

In [ ]:
if RUN_TEARDOWN:
    client.beta.sessions.archive(session.id, betas=[BETA])
    client.beta.agents.archive(agent.id, betas=[BETA])
    print("Torn down: session and agent archived.")
else:
    print("Teardown SKIPPED - agent and session are still LIVE.")
    print("Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.")
    print("  agent:  ", agent.id)
    print("  session:", session.id)

## 7. Self-check: domain-weighted questions

**On Domain 3 (Claude Code, 20%).** This capstone does not fake Domain 3. Domain 3 is CLI and configuration - CLAUDE.md, settings, hooks, MCP wiring - and it is taught live in the **segment-2 course notebook**, not in this API surface. The one honest bridge to what you built here: an agent's **`system`** prompt is a **versioned artifact** you check in and change on purpose, exactly like **CLAUDE.md** is a versioned artifact for Claude Code. Same discipline, different surface.

The questions below are **original** to this course, weighted toward the domains this capstone exercised. Read the situation, pick one option, then check the answer and the per-distractor reasoning.

---

**Q1 (D1 - Agentic, 27%).** Your streaming loop over `sessions.events.stream` prints the first `agent.message` and returns immediately. Users complain the agent "answers before it looks things up." What is the fix?

- **A.** Add a `time.sleep(2)` before returning so the tool has time to run.
- **B.** Keep streaming and only stop on **`session.status_idle`**; a mid-turn `agent.message` is not the end of the turn.
- **C.** Resend the full user turn on every iteration so context is not lost.
- **D.** Switch the model to Sonnet so it stops emitting early messages.

**Answer: B.** The turn is done when the session goes idle, and `stop_reason` on that event tells you why. **A** guesses at timing instead of reading the signal the API gives you. **C** breaks server-side session memory and double-bills the turn. **D** confuses model choice with loop control - any model streams interim messages.

---

**Q2 (D2 - Tools, 18%).** The agent requested your `venue_capacity` **custom** tool. You send back an event of type `user.tool_result` carrying a `tool_use_id`. The turn stalls. Why?

- **A.** Custom tools are answered with **`user.custom_tool_result`** keyed by **`custom_tool_use_id`**; `user.tool_result` with `tool_use_id` is for built-in and MCP tools.
- **B.** You must archive the session and recreate it before answering any tool.
- **C.** Tool results have to be sent as a new `user.message` text block, not a tool-result event.
- **D.** The capacity number was wrong, so the API rejected the event.

**Answer: A.** Two distinct event types with two distinct id fields; matching custom tools to the built-in field leaves the tool_use unanswered. **B** is teardown, not tool answering, and would destroy the turn. **C** is the pre-tool-use pattern and would not satisfy the pending tool_use. **D** confuses a wrong value (a business error) with a protocol error - a wrong number still lands as a valid result.

---

**Q3 (D4 - Structured output, 20%).** Your prompt asks for triage JSON with a `refund_owed` integer field. The incident report never mentions refunds. The agent returns `"refund_owed": 0`. Your validator flags it. What behavior should the `system` prompt enforce?

- **A.** Default every unstated numeric field to `0` so the JSON always parses.
- **B.** Return **`null`** for any field the source does not state, and never invent a value.
- **C.** Omit unstated keys entirely so the object stays small.
- **D.** Return the whole object as a string and let the caller parse it.

**Answer: B.** `0` is a claim the report never made; `null` is the honest "not stated." **A** manufactures a false refund of zero that downstream logic may treat as a real decision. **C** breaks a fixed schema - consumers expect the key to be present. **D** defeats structured output and pushes parsing risk onto every caller.

---

**Q4 (D5 - Context and escalation, 15%).** Two reports arrive. Report one, shouted: "The 9pm show is SOLD OUT and I can't find a seat!" Report two, said flatly: "By the way, the east exit is chained during the sold-out block." Which escalates to a human, and on what basis?

- **A.** Report one, because it was the loudest and most urgent-sounding.
- **B.** Report two, because a blocked fire exit is a **safety-shape** incident regardless of the calm tone; a sold-out seating question is self-serve.
- **C.** Both, because any mention of "sold out" is a revenue risk.
- **D.** Neither, because neither report used the word "emergency."

**Answer: B.** Escalation routes on incident **shape**, not on volume or sentiment. **A** rewards loudness and would bury the real hazard. **C** over-escalates a routine seating question and floods the human queue. **D** keyword-matches on "emergency" and misses a blocked exit that never used the word.

---

**Q5 (D1 + teardown, 27%).** A cell mid-notebook raised an exception, so your teardown cell never ran. What is the consequence, and the durable fix?

- **A.** Nothing - archiving is optional and sessions expire on their own instantly.
- **B.** A live **session** is a billable runtime that keeps costing until archived; wrap the run in **`try/finally`** so teardown always executes, and use **`archive`** (there is no `agents.delete`).
- **C.** The agent is deleted automatically when the kernel restarts.
- **D.** You must call `environments.delete` first or archive will fail.

**Answer: B.** Leaving a session open leaks cost; `try/finally` guarantees the archive path runs on both success and failure. **A** treats a billable runtime as free and self-cleaning. **C** ties API resource lifecycle to a local kernel, which does not hold. **D** invents an ordering constraint - archive the session and the agent directly; the environment is separate.

---

**Q6 (D2 + D4, mixed).** Your `venue_capacity` tool call fails transiently. The agent, lacking a number, writes `"capacity": 2362` from memory of a past show. Which two habits from this capstone would have prevented the fabricated number?

- **A.** A `system` rule to **never guess capacity and always call the tool**, plus a structured-error tool result so the agent retries instead of inventing.
- **B.** Raising `max_tokens` so the agent has room to reason its way to the right number.
- **C.** Lowering temperature to 0 so the model is deterministic about the wrong number.
- **D.** Switching the tool from custom to built-in so it cannot fail.

**Answer: A.** The prompt discipline stops the guess and the structured error (an `is_error` tool result) gives the agent a signal to retry rather than hallucinate. **B** gives more room to invent, not more accuracy. **C** makes a wrong answer repeatable, not correct. **D** does not exist as a lever - a capacity lookup is inherently a custom tool, and built-in tools fail too.